# Fase 3 · Feature Engineering (Ingeniería de características)
**Dataset de propiedades en venta — Mérida, Yucatán**

## Propósito
Transformar el dataset crudo validado en la Fase 2 en un conjunto listo para
modelar: eliminar duplicados, derivar variables binarias del campo `notas`,
preparar la variable objetivo y dejar definido el protocolo de evaluación
(partición estratificada / validación cruzada) que utilizarán los notebooks
de modelado.

## Entrada y salida
- **Entrada:** `data/propiedades.csv` — 60 propiedades en venta de 6 colonias,
  recolectadas manualmente de Inmuebles24 (Fase 1) y analizadas en
  `01_eda_inicial.ipynb` (Fase 2).
- **Salida:** conjunto de modelado con las características construidas y la
  partición/CV definida, para `03_modelo_baseline.ipynb` y
  `04_modelo_comparativo.ipynb`.

## Nota sobre el tamaño de muestra
Con n=60, este trabajo es una prueba de concepto metodológica, no un modelo de
valuación desplegable. Las decisiones priorizan un flujo sin fuga de información
y estimaciones honestas por validación cruzada por encima de la métrica puntual.

## Contenido (provisional)
1. Configuración
2. Carga y validación del contrato de datos
3. Construcción del conjunto de modelado (eliminación de duplicados)
4. Ingeniería de características desde `notas`
5. Preparación de la variable objetivo
6. Protocolo de evaluación (partición estratificada / CV)
7. Prueba estadística de preventa
8. Exportación del conjunto de modelado

## 1. Configuración
Importamos las librerías y definimos la ruta a los datos. La ruta se construye
a partir de la raíz del proyecto, de modo que el notebook corre igual desde
cualquier carpeta o máquina.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)  # ver todas las columnas al inspeccionar

# El notebook vive en notebooks/; los datos en data/ (un nivel arriba).
# Resolvemos la raíz para que la ruta no dependa del directorio de ejecución.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "propiedades.csv"

print("Raíz del proyecto:", PROJECT_ROOT)
print("¿Existe el CSV?:", DATA_PATH.exists())

Raíz del proyecto: /Users/edson/Documents/GitHub/dataset-inmuebles-merida
¿Existe el CSV?: True


## 2. Carga y validación del contrato de datos
Antes de transformar algo, se confirma que el archivo real coincide con el
contrato documentado (forma, columnas, tipos, faltantes) y se inspecciona la
estructura categórica que condicionará la estratificación y el test de preventa.
Celda de solo lectura: no modifica los datos.

In [2]:
# Validación del contrato de datos (solo lectura, no modifica nada)
COLUMNAS_ESPERADAS = [
    "fecha_registro", "url", "url_archivo", "operación", "tipo_inmueble",
    "colonia", "precio", "m2_construccion", "m2_terreno", "recamaras",
    "banos", "estacionamientos", "antigüedad", "es_preventa", "notas",
]
COLUMNAS_NUMERICAS = [
    "precio", "m2_construccion", "m2_terreno", "recamaras",
    "banos", "estacionamientos", "antigüedad",
]

df = pd.read_csv(DATA_PATH)

print("Forma:", df.shape)  # esperado: (60, 15)
print("Columnas en orden esperado:", list(df.columns) == COLUMNAS_ESPERADAS)

print("\nTipos inferidos:")
print(df.dtypes)

print("\nFaltantes por columna:")
print(df.isna().sum())

print("\nCeros en columnas numéricas (revisar si alguno debería ser NaN):")
print((df[COLUMNAS_NUMERICAS] == 0).sum())

print("\nPor colonia:")
print(df["colonia"].value_counts())
print("\nPor tipo_inmueble:")
print(df["tipo_inmueble"].value_counts())
print("\nPor es_preventa:")
print(df["es_preventa"].value_counts())

mask_dup = df["notas"].str.contains("posible duplicado", case=False, na=False)
print(f"\nFilas marcadas como posible duplicado: {int(mask_dup.sum())}")
print(df.loc[mask_dup, ["colonia", "precio", "notas"]])

Forma: (60, 15)
Columnas en orden esperado: True

Tipos inferidos:
fecha_registro          str
url                     str
url_archivo         float64
operación               str
tipo_inmueble           str
colonia                 str
precio                int64
m2_construccion       int64
m2_terreno            int64
recamaras             int64
banos                 int64
estacionamientos    float64
antigüedad          float64
es_preventa             str
notas                   str
dtype: object

Faltantes por columna:
fecha_registro       0
url                  0
url_archivo         60
operación            0
tipo_inmueble        0
colonia              0
precio               0
m2_construccion      0
m2_terreno           0
recamaras            0
banos                0
estacionamientos     1
antigüedad          39
es_preventa          0
notas                1
dtype: int64

Ceros en columnas numéricas (revisar si alguno debería ser NaN):
precio              0
m2_construccion     0
m2_terr

In [3]:
pd.set_option("display.max_colwidth", None)
sgc = df[df["colonia"] == "santa gertrudis copo"].sort_values("precio", ascending=False)
print(sgc[["precio", "m2_construccion", "m2_terreno", "recamaras", "banos", "notas"]])

      precio  m2_construccion  m2_terreno  recamaras  banos  \
15  16900000              542        1346          3      4   
18  16900000              542        1346          3      4   
19   6400000              252         293          3      3   
17   5900000              253         414          5      4   
11   5500000              283         283          2      3   
16   4300000              220         250          3      4   
12   3360000              200         227          3      2   
13   2900000              102         102          2      2   
10   1950000               59          59          1      1   
14   1790000               42          42          2      2   

                                                                                                                                                         notas  
15                                                                                 alberca con jacuzzi, bar, cuarto de servicio, bodega, una plan

## 3. Construcción del conjunto de modelado: eliminación de duplicados
Se construye el conjunto de modelado descartando los registros marcados como
"posible duplicado" en `notas`. Se identifican por esa marca —identificador
durable (es decir, que sobrevive a reordenamientos y reindexados) definido en la Fase 1—, no por índice de fila. La fila 18 se confirmó
como copia de la fila 15 (misma propiedad: precio, superficie, recámaras y
baños idénticos). Se descarta del conjunto **completo**, no solo del de
entrenamiento: si la copia quedara en test y su original en entrenamiento, la
métrica se inflaría con un punto casi memorizado (fuga), aparte del doble
conteo de una misma propiedad.

In [4]:
# Conjunto de modelado: se descartan los duplicados marcados en `notas`
mask_dup = df["notas"].str.contains("posible duplicado", case=False, na=False)
df_model = df[~mask_dup].copy()

print(f"Descartadas (posible duplicado): {int(mask_dup.sum())}")
print(f"Crudo: {df.shape[0]} filas  →  modelado: {df_model.shape[0]} filas")

# Verificación del par conocido: la copia (18) se fue, la original (15) permanece
print("Original (índice 15) presente:", 15 in df_model.index)
print("Copia (índice 18) descartada:", 18 not in df_model.index)

Descartadas (posible duplicado): 1
Crudo: 60 filas  →  modelado: 59 filas
Original (índice 15) presente: True
Copia (índice 18) descartada: True


## 4. Ingeniería de características desde `notas`
Se derivan cuatro variables binarias del texto libre de `notas` —las únicas con
frecuencia suficiente (>10%) para tener potencia con n≈59—. Se construyen por
coincidencia de palabras clave, con dos cuidados de vocabulario:
- `tiene_piscina` busca "piscina" y "alberca": en español de México son
  sinónimos y el dataset usa ambas; matchear solo una subcontaría.
- `tiene_mantenimiento_con_monto` busca "mantenimiento $" (cuota con monto
  explícito), no "mantenimiento" a secas, que también aparece en negaciones.
Las features borderline (<10%) quedan fuera del modelo por insuficiencia
muestral, como concluyó el EDA.

In [5]:
# Sección 4: variables binarias desde `notas`
notas = df_model["notas"].fillna("").str.lower()

df_model["tiene_piscina"]                 = notas.str.contains("piscina|alberca", regex=True).astype(int)
df_model["tiene_cuarto_servicio"]         = notas.str.contains("cuarto de servicio", regex=False).astype(int)
df_model["es_una_planta"]                 = notas.str.contains("una planta", regex=False).astype(int)
df_model["tiene_mantenimiento_con_monto"] = notas.str.contains(r"mantenimiento\s*\$", regex=True).astype(int)

features_notas = [
    "tiene_piscina", "tiene_cuarto_servicio",
    "es_una_planta", "tiene_mantenimiento_con_monto",
]
print("Conteos por feature (df_model, 59 filas):")
print(df_model[features_notas].sum())

print("\nDesglose piscina vs alberca:")
print("  solo 'piscina':", (notas.str.contains("piscina") & ~notas.str.contains("alberca")).sum())
print("  solo 'alberca':", (notas.str.contains("alberca") & ~notas.str.contains("piscina")).sum())
print("  ambas:          ", (notas.str.contains("piscina") & notas.str.contains("alberca")).sum())

Conteos por feature (df_model, 59 filas):
tiene_piscina                    37
tiene_cuarto_servicio            14
es_una_planta                     9
tiene_mantenimiento_con_monto     8
dtype: int64

Desglose piscina vs alberca:
  solo 'piscina': 26
  solo 'alberca': 11
  ambas:           0


## 5. Preparación de la variable objetivo
Se aplica la transformación logarítmica a `precio` para obtener la variable
objetivo del modelo lineal. El EDA confirmó sesgo positivo marcado en la
distribución del precio (cola derecha larga); `log(precio)` comprime esa
asimetría y estabiliza la varianza —supuesto necesario para regresión lineal—.
Los modelos de árbol (Random Forest, Gradient Boosting) no requieren la
transformación, pero se aplica consistentemente para mantener la variable
objetivo uniforme entre modelos y facilitar la comparación de métricas en
escala original vía `exp(predicción)`.

In [6]:
df_model["log_precio"] = np.log(df_model["precio"])

print("Estadísticos — precio original vs log(precio):")
print(df_model[["precio", "log_precio"]].describe().round(2))

print(f"\nRango precio original: {df_model['precio'].min():,} — {df_model['precio'].max():,}")
print(f"Rango log(precio):     {df_model['log_precio'].min():.3f} — {df_model['log_precio'].max():.3f}")

Estadísticos — precio original vs log(precio):
            precio  log_precio
count        59.00       59.00
mean    8015011.97       15.52
std     8760147.08        0.80
min     1790000.00       14.40
25%     2900000.00       14.88
50%     4785000.00       15.38
75%     8000000.00       15.89
max    41000000.00       17.53

Rango precio original: 1,790,000 — 41,000,000
Rango log(precio):     14.398 — 17.529


## 6. Protocolo de evaluación

### Fundamento
Con n=59, un holdout único 80/20 deja ~12 puntos de prueba. Un RMSE estimado
sobre 12 observaciones tiene varianza tan alta que es estadísticamente poco
informativo: el intervalo de confianza de la estimación se solaparía entre
modelos distintos. La validación cruzada es, en este tamaño muestral, la única
forma de obtener una estimación razonablemente estable del error de
generalización.

### Estratificación por colonia
El dataset tiene tres segmentos de precio muy dispares
(YCC ~16.5M · medio ~5M · accesible ~3M) y solo 9–11 propiedades por colonia.
Con folds aleatorios, un fold puede quedar sin representación de algún
segmento —p. ej. sin ninguna propiedad de YCC en entrenamiento—, lo que
desestabiliza y sesga la estimación. Estratificar por colonia garantiza que
cada fold es un microcosmos representativo del dataset.

### Elección de k y repeticiones (heurística)
Se usa k=5 con repetición (repeated stratified k-fold). Con 59 filas y 6
estratos, k=5 deja ~47 para entrenamiento y ~12 para validación por fold —
razonable dado el tamaño. La repetición promedia sobre distintas
aleatorizaciones del mismo fold y reduce la varianza de la estimación del
error. La elección de k=5 y el número de repeticiones son decisiones
heurísticas, no formalmente optimizadas: se etiquetan como tales.

### CV anidada para modelos con hiperparámetros
El baseline OLS tiene un conjunto de features fijo y sin hiperparámetros que
ajustar: un k-fold estratificado simple es suficiente y honesto para él.
Para Random Forest y Gradient Boosting, tunear hiperparámetros en el mismo
fold que reporta desempeño infla la métrica (selección sobre el conjunto de
validación). Se requiere CV anidada: loop externo para estimar el error de
generalización, loop interno para seleccionar hiperparámetros.

### Métricas
RMSE y MAE en escala original (revertiendo `exp(predicción)`), para que los
errores sean interpretables en pesos mexicanos.

In [7]:
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold

# Estratificador por colonia — baseline OLS
cv_baseline = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Estratificador por colonia — loop externo para RF/GB (CV anidada)
cv_outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

# Vector de estratificación
strat_col = df_model["colonia"]

print("Distribución por colonia (conjunto de modelado):")
print(strat_col.value_counts())

print("\nVerificación — tamaños de fold (cv_baseline):")
for fold, (train_idx, val_idx) in enumerate(cv_baseline.split(df_model, strat_col), start=1):
    print(f"  Fold {fold}: train={len(train_idx)}, val={len(val_idx)}")

print("\nProtocolo listo.")
print(f"  Baseline OLS  → StratifiedKFold(k=5)")
print(f"  RF / GB       → RepeatedStratifiedKFold(k=5, repeticiones=10), loop externo")

Distribución por colonia (conjunto de modelado):
colonia
yucatan country club    11
altabrisa               10
garcia gineres          10
chuburna de hidalgo     10
santa gertrudis copo     9
francisco de montejo     9
Name: count, dtype: int64

Verificación — tamaños de fold (cv_baseline):
  Fold 1: train=47, val=12
  Fold 2: train=47, val=12
  Fold 3: train=47, val=12
  Fold 4: train=47, val=12
  Fold 5: train=48, val=11

Protocolo listo.
  Baseline OLS  → StratifiedKFold(k=5)
  RF / GB       → RepeatedStratifiedKFold(k=5, repeticiones=10), loop externo


## 7. Prueba estadística: efecto de preventa sobre el precio

Se evalúa si el precio difiere significativamente entre propiedades en preventa
(n=10) y terminadas (n=49). El resultado decide si `es_preventa` entra al
modelo como variable binaria.

Se usa Mann-Whitney U en vez de test t por dos razones:
1. Con n=10 en el grupo de preventa no es razonable asumir normalidad.
2. Mann-Whitney compara distribuciones completas (cuál grupo tiende a valores
   más altos), no solo medias — más adecuado para una variable con la
   dispersión que tiene `log_precio`.

Se aplica sobre `log_precio`, consistente con la variable objetivo del modelo.
Un resultado significativo (p < 0.05) indica que las preventas tienen una
dinámica de precio distinta y la variable debe entrar al modelo. Un resultado
no significativo no prueba ausencia de efecto: con n=10 el test tiene potencia
limitada para detectar diferencias moderadas.

In [8]:
from scipy.stats import mannwhitneyu

preventa_si = df_model.loc[df_model["es_preventa"] == "si", "log_precio"]
preventa_no = df_model.loc[df_model["es_preventa"] == "no", "log_precio"]

print(f"Grupo preventa:   n={len(preventa_si)}, mediana log_precio={preventa_si.median():.3f} "
      f"(≈ {int(round(preventa_si.median()**2/1e6 if False else __import__('numpy').exp(preventa_si.median())/1e6))}M MXN)")
print(f"Grupo terminado:  n={len(preventa_no)}, mediana log_precio={preventa_no.median():.3f} "
      f"(≈ {int(round(__import__('numpy').exp(preventa_no.median())/1e6))}M MXN)")

stat, p_value = mannwhitneyu(preventa_si, preventa_no, alternative="two-sided")

print(f"\nMann-Whitney U = {stat:.1f},  p = {p_value:.4f}")

if p_value < 0.05:
    print("→ Diferencia significativa (p < 0.05): incluir es_preventa como feature.")
else:
    print("→ Diferencia no significativa (p ≥ 0.05): incluir es_preventa de todas formas,")
    print("  con la advertencia de que el test tiene potencia limitada con n=10.")

Grupo preventa:   n=10, mediana log_precio=15.045 (≈ 3M MXN)
Grupo terminado:  n=49, mediana log_precio=15.483 (≈ 5M MXN)

Mann-Whitney U = 127.0,  p = 0.0176
→ Diferencia significativa (p < 0.05): incluir es_preventa como feature.


## 8. Preparación final y exportación

### 8.1 Codificación y limpieza antes de exportar
Tres transformaciones antes de serializar el conjunto de modelado:
- `es_preventa` (si/no → 1/0): resultado del test de Mann-Whitney; entra
  como feature binaria.
- `es_casa` desde `tipo_inmueble` (casa → 1, departamento → 0): encoding
  binario directo; con solo dos categorías no se pierde información.
- `estacionamientos`: único faltante del dataset (1/59). Se imputa con la
  mediana del conjunto de modelado — decisión heurística y defensible a esta
  escala (imputa un solo punto con el valor central observado).

`colonia` se exporta como cadena de texto: su encoding (one-hot con
`drop_first` para OLS, sin restricción para RF/GB) es una decisión
model-específica que se toma en los notebooks de modelado.
La multicolinealidad `m2_construccion` ↔ `m2_terreno` (r=0.95) también se
resuelve allá, donde se puede medir el impacto empíricamente con CV.

In [9]:
# --- Codificación de categóricas estables ---
df_model["es_preventa"] = (df_model["es_preventa"] == "si").astype(int)
df_model["es_casa"]     = (df_model["tipo_inmueble"] == "casa").astype(int)

# --- Imputación de estacionamientos (1 faltante) ---
median_est    = df_model["estacionamientos"].median()
n_missing_est = df_model["estacionamientos"].isna().sum()
df_model["estacionamientos"] = (
    df_model["estacionamientos"]
    .fillna(median_est)
    .astype(int)
)

print("Codificación:")
print(f"  es_preventa  → {df_model['es_preventa'].value_counts().to_dict()}")
print(f"  es_casa      → {df_model['es_casa'].value_counts().to_dict()}")

print(f"\nImputación estacionamientos:")
print(f"  Faltantes imputados: {n_missing_est}  |  valor usado: {median_est}")
print(f"  Distribución resultante:\n{df_model['estacionamientos'].value_counts().sort_index()}")

print(f"\nFaltantes restantes en columnas de modelado:")
cols_modelo = [
    "log_precio", "m2_construccion", "m2_terreno", "recamaras",
    "banos", "estacionamientos", "es_preventa", "es_casa",
    "tiene_piscina", "tiene_cuarto_servicio",
    "es_una_planta", "tiene_mantenimiento_con_monto", "colonia",
]
print(df_model[cols_modelo].isna().sum())

Codificación:
  es_preventa  → {0: 49, 1: 10}
  es_casa      → {1: 43, 0: 16}

Imputación estacionamientos:
  Faltantes imputados: 1  |  valor usado: 2.0
  Distribución resultante:
estacionamientos
1    16
2    31
3     2
4     9
6     1
Name: count, dtype: int64

Faltantes restantes en columnas de modelado:
log_precio                       0
m2_construccion                  0
m2_terreno                       0
recamaras                        0
banos                            0
estacionamientos                 0
es_preventa                      0
es_casa                          0
tiene_piscina                    0
tiene_cuarto_servicio            0
es_una_planta                    0
tiene_mantenimiento_con_monto    0
colonia                          0
dtype: int64


### 8.2 Exportación
Se serializa `df_model` completo — todas las columnas originales más las
features construidas — para que los notebooks de modelado puedan seleccionar
las que necesiten. El archivo vive en `data/` junto al CSV crudo, con nombre
que distingue su propósito.

In [10]:
OUTPUT_PATH = PROJECT_ROOT / "data" / "propiedades_modelo.csv"
df_model.to_csv(OUTPUT_PATH, index=False)

print(f"Exportado: {OUTPUT_PATH}")
print(f"Forma: {df_model.shape}")
print(f"\nColumnas exportadas:")
print(list(df_model.columns))

Exportado: /Users/edson/Documents/GitHub/dataset-inmuebles-merida/data/propiedades_modelo.csv
Forma: (59, 21)

Columnas exportadas:
['fecha_registro', 'url', 'url_archivo', 'operación', 'tipo_inmueble', 'colonia', 'precio', 'm2_construccion', 'm2_terreno', 'recamaras', 'banos', 'estacionamientos', 'antigüedad', 'es_preventa', 'notas', 'tiene_piscina', 'tiene_cuarto_servicio', 'es_una_planta', 'tiene_mantenimiento_con_monto', 'log_precio', 'es_casa']
